In [59]:
import re
import html
import unicodedata

import pandas as pd
import numpy as np


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

In [60]:
phishing_sms = pd.read_csv('datasets/phishing_dataset_with_category.csv')

In [61]:
phishing_sms.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   text      1000 non-null   str  
 1   category  1000 non-null   str  
 2   label     1000 non-null   str  
dtypes: str(3)
memory usage: 123.7 KB


In [62]:
phishing_sms.head()

,text,category,label
0,Warning: Unusual login attempt detected on you...,urgency,phishing
1,Urgent! Your Google has been compromised. Clic...,urgency,phishing
2,This is an official notice from Amazon. Your a...,authority,phishing
3,"As per HMRC regulations, you must update your ...",authority,phishing
4,Immediate action required: Your Spotify subscr...,urgency,phishing


In [63]:
phishing_sms['label'].value_counts()

label
phishing    1000
Name: count, dtype: int64

In [64]:
def normalize_urls(text):
    return re.sub(r"(https?://\S+|www\.\S+)", " <URL> ", text)


def normalize_emails(text):
    return re.sub(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}\b", " <EMAIL> ", text)


def normalize_numbers(text):
    return re.sub(r"\b\d+(?:[\.,:/-]\d+)*\b", " <NUM> ", text)


def remove_html_tags(text):
    return re.sub(r"<[^>]+>", " ", text)

In [65]:
def clean_text(text):
    if pd.isna(text):
        return ""

    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    
    text = remove_html_tags(text)
    
    text = normalize_urls(text)
    
    text = normalize_emails(text)
    
    text = normalize_numbers(text)

    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [66]:
phishing_sms['text'] = phishing_sms['text'].apply(clean_text)

In [67]:
phishing_sms.duplicated(subset=["text"])

0      False
1      False
2      False
3      False
4      False
       ...  
995     True
996     True
997     True
998     True
999     True
Length: 1000, dtype: bool

In [68]:
phishing_sms = phishing_sms.drop_duplicates(subset=["text"]).reset_index(drop=True)

In [69]:
phishing_sms['category'].value_counts()

category
persuasion    29
urgency       25
authority     25
Name: count, dtype: int64

In [73]:
pipeline_sms = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "clf",
        LogisticRegression(
            max_iter=2000,
            random_state=42,
            class_weight="balanced"
        )
    )
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [75]:
X = phishing_sms["text"]
Y = phishing_sms["category"]

full_pred = []
full_true = []

target_names = ["authority", "persuasion", "urgency"]

In [78]:
fold=1

for training_index, testing_index in cv.split(X, Y):

    X_train, X_test = X[training_index], X[testing_index]
    Y_train, Y_test = Y[training_index], Y[testing_index]

    pipeline_sms.fit(X_train, Y_train)

    predictions = pipeline_sms.predict(X_test)

    print(f"\n===== Fold {fold} =====")
    print(classification_report(
        Y_test,
        predictions,
        target_names=target_names,
        zero_division=0
    ))
    print("Confusion Matrix:")
    print(confusion_matrix(Y_test, predictions))

    full_true.extend(Y_test)
    full_pred.extend(predictions)

    fold += 1


===== Fold 1 =====
              precision    recall  f1-score   support

   authority       1.00      1.00      1.00         5
  persuasion       1.00      1.00      1.00         6
     urgency       1.00      1.00      1.00         5

    accuracy                           1.00        16
   macro avg       1.00      1.00      1.00        16
weighted avg       1.00      1.00      1.00        16

Confusion Matrix:
[[5 0 0]
 [0 6 0]
 [0 0 5]]

===== Fold 2 =====
              precision    recall  f1-score   support

   authority       1.00      1.00      1.00         5
  persuasion       1.00      1.00      1.00         6
     urgency       1.00      1.00      1.00         5

    accuracy                           1.00        16
   macro avg       1.00      1.00      1.00        16
weighted avg       1.00      1.00      1.00        16

Confusion Matrix:
[[5 0 0]
 [0 6 0]
 [0 0 5]]

===== Fold 3 =====
              precision    recall  f1-score   support

   authority       1.00      1.

In [79]:

print("\n===== Overall Classification Report Across All Folds =====")
print(classification_report(
    full_true,
    full_pred,
    target_names=target_names,
    zero_division=0
))





===== Overall Classification Report Across All Folds =====
              precision    recall  f1-score   support

   authority       1.00      1.00      1.00        25
  persuasion       1.00      1.00      1.00        29
     urgency       1.00      1.00      1.00        25

    accuracy                           1.00        79
   macro avg       1.00      1.00      1.00        79
weighted avg       1.00      1.00      1.00        79



In [80]:
print("Overall Confusion Matrix:")
print(confusion_matrix(full_true, full_pred))

Overall Confusion Matrix:
[[25  0  0]
 [ 0 29  0]
 [ 0  0 25]]


In [88]:
print(phishing_sms[phishing_sms['category']== 'persuasion'].head(20)['text'].to_list())

['you’ve been selected for a premium cloud storage trial. sign up now before the offer expires.', 'you’ve been selected for a premium email trial. sign up now before the offer expires.', 'special access granted! login now to receive your free e-book before it’s too late.', 'congratulations! you have won a free vacation. click here to claim your reward.', 'you’ve been selected for a premium banking trial. sign up now before the offer expires.', 'claim your limited-time buy-one-get-one-free offer on software subscription. click here to activate your reward.', 'claim your limited-time <num> % discount on software subscription. click here to activate your reward.', 'you’ve been selected for a premium netflix trial. sign up now before the offer expires.', 'special access granted! login now to receive your cashback before it’s too late.', 'special access granted! login now to receive your gift card before it’s too late.', 'claim your limited-time <num> % discount on smartphone. click here to